# Train Shared Vision Backbone (Ball + Marker CNN)
This notebook clones the repository, extracts the merged `shared_vision` gold dataset from your Google Drive, and trains the ~70K-param Shared Encoder Backbone (ball regression head + marker segmentation head + marker heatmap head) defined in `train_cnn_2d_tracker_marker.py`.

In [ ]:
DATASET_NAME = 'shared_vision'
VERSION = 'v1'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!rm -rf /content/ball_balance_video_controlled
!git clone https://github.com/Jack0468/ball_balance_video_controlled.git
!pip install pandas torch torchvision albumentations opencv-python-headless matplotlib
import torch
print(f"Setup complete. Using torch {torch.__version__} ({torch.cuda.get_device_properties(0).name if torch.cuda.is_available() else 'CPU'})")

In [ ]:
# Unzip the merged 03_gold shared_vision dataset (images/ + masks/ + labels.csv)
!mkdir -p /content/ball_balance_video_controlled/host_software/data/03_gold
!unzip -q -o /content/drive/MyDrive/{DATASET_NAME}.zip -d /content/ball_balance_video_controlled/host_software/data/03_gold
print("Dataset unzipped!")

### Training

In [ ]:
%cd /content/ball_balance_video_controlled
# Run as a module (-m), not a plain script -- the trainer imports
# host_software.ml_vision.training.{shared_vision_dataset,augmentations} as package-qualified
# paths, which only resolve when the repo root is on sys.path (i.e. invoked via -m from here).
!python -m host_software.ml_vision.training.train_cnn_2d_tracker_marker \
    --csv-file host_software/data/03_gold/{DATASET_NAME}/labels.csv \
    --images-dir host_software/data/03_gold/{DATASET_NAME}/images \
    --mask-dir host_software/data/03_gold/{DATASET_NAME}/masks \
    --output-dir /content/drive/MyDrive/VRI_Models/shared_vision_backbone_{VERSION}

### Results
`train_cnn_2d_tracker_marker.py` already exports the best checkpoint to ONNX and writes a per-session validation breakdown and loss curve -- there's no separate `evaluate_*` script for this model yet, so this just surfaces those artifacts.

In [ ]:
from IPython.display import Image, display
import pandas as pd

output_dir = f'/content/drive/MyDrive/VRI_Models/shared_vision_backbone_{VERSION}'
display(Image(filename=f'{output_dir}/training_curve.png'))
pd.read_csv(f'{output_dir}/per_session_eval.csv')